# J4-03 · Prompts versionnés : sortir les prompts du code

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 4 · créneau 09h15–09h30 · ⏱️ 15 min (**démo formateur** : labels, retour arrière ; le notebook se refait en autonomie)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Externaliser un prompt dans Langfuse, avec ses **variables** · 2. **Versionner** et poser des **labels** (`production`, `staging`) · 3. **Lier** chaque génération à la version de prompt utilisée · 4. **Promouvoir** une version après mesure, et **revenir en arrière** · 5. Prévoir un **repli** si Langfuse est injoignable |
| 🧱 **Fil rouge** | Le prompt de tri des tickets (J1) classe le VPN et le Wi-Fi hors « réseau » (constaté en J2-03) : on le corrige, version par version |
| ✅ **Prérequis** | J4-01, J4-02 |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Pourquoi sortir les prompts du code ?

### [Théorie] Le prompt est une configuration qui change plus vite que le code

| Prompt dans le code | Prompt dans Langfuse |
|---|---|
| changer une virgule = un déploiement | nouvelle **version**, sans redéployer |
| « quelle version tournait mardi ? » : l'historique Git, si on le retrouve | chaque génération est **liée** à sa version |
| A/B entre deux formulations : du code à écrire | deux **labels**, deux runs d'expérience (J4-05) |
| retour arrière : un nouveau déploiement | déplacer le label `production` |

> ⚠️ **Le revers :** un prompt modifiable sans déploiement est un prompt modifiable **sans revue**. Qui peut promouvoir en `production` ? Avec quel test ? On répond à ces questions en fin de notebook (gouvernance) ; l'automatisation par une gate de CI est hors programme de cette session.

## 3. Créer la version 1 (celle du jour 1)

Variables entre doubles accolades : `{{categories}}`, `{{ticket}}`.

In [ ]:
NOM = "helpdesk-tri"
CATEGORIES = "reseau, materiel, logiciel, acces, autre"

v1 = langfuse.create_prompt(
    name=NOM, type="chat", labels=["production"],
    prompt=[{"role": "system", "content": "Tu es l'agent de tri N1 du help-desk de NordTech SA. Classe le ticket dans UNE "
                                          "catégorie parmi : {{categories}}. Réponds uniquement par le nom de la catégorie, "
                                          "en minuscules, sans ponctuation."},
            {"role": "user", "content": "Ticket : {{ticket}}"}],
    commit_message="v1 : prompt du jour 1",
)
print(f"version {v1.version} · labels {v1.labels}")
print(langfuse.get_prompt(NOM, type="chat", label="production").compile(categories=CATEGORIES, ticket="VPN coupé"))

## 4. Utiliser le prompt depuis LangChain, et le lier aux traces

`get_langchain_prompt()` convertit les `{{variables}}` en `{variables}` LangChain. En plaçant le prompt Langfuse dans les **métadonnées** du `ChatPromptTemplate`, le `CallbackHandler` lie chaque génération à **cette version** : dans l'interface, la page du prompt affiche ses générations, leur latence et leurs scores.

In [ ]:
import time

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from helpdesk_ops.tri import normaliser_categorie


def classifieur_depuis(prompt_langfuse):
    modele = ChatPromptTemplate.from_messages(prompt_langfuse.get_langchain_prompt())
    modele.metadata = {"langfuse_prompt": prompt_langfuse}   # ← lien génération ↔ version du prompt
    return modele | llm | StrOutputParser()


# Six tickets du dataset de J4-05 dont la catégorie a posé problème pendant la préparation
CAS = [("Comment configurer le VPN depuis l'hôtel ?", "reseau"),
       ("Le Wi-Fi de la salle B coupe toutes les 5 minutes", "reseau"),
       ("L'ERP est inaccessible pour toute la comptabilité depuis 9 h", "logiciel"),
       ("Je voudrais une souris sans fil", "materiel"),
       ("J'ai cliqué sur le lien d'un mail suspect et j'ai saisi mon mot de passe", "acces"),
       ("J'ai perdu mon téléphone, je ne peux plus valider le MFA", "acces")]


def evaluer(prompt_langfuse) -> float:
    classifieur = classifieur_depuis(prompt_langfuse)
    justes = 0
    with propagate_attributes(session_id="j4-03-prompts", tags=["jour-4", "prompts"], version=f"{NOM}-v{prompt_langfuse.version}"):
        for ticket, attendu in CAS:
            obtenu = normaliser_categorie(classifieur.invoke({"categories": CATEGORIES, "ticket": ticket},
                                                             config={"callbacks": [langfuse_handler], "run_name": "tri"}))
            justes += obtenu == attendu
            print(f"   {'✅' if obtenu == attendu else '❌'} {obtenu:<9} (attendu {attendu:<8}) ← {ticket[:60]}")
    return justes / len(CAS)


print(f"v{v1.version} → justesse {evaluer(langfuse.get_prompt(NOM, type='chat', label='production')):.0%}")
langfuse.flush()

## 5. Version 2, en `staging` : des définitions pour chaque catégorie

On ne touche pas à la production : la v2 reçoit le label `staging`, on la **mesure** sur les mêmes cas.

In [ ]:
v2 = langfuse.create_prompt(
    name=NOM, type="chat", labels=["staging"],
    prompt=[{"role": "system", "content": "Tu es l'agent de tri N1 du help-desk de NordTech SA. Classe le ticket dans UNE "
                                          "catégorie parmi : {{categories}}.\n"
                                          "- reseau : VPN, Wi-Fi, connexion, serveur injoignable\n"
                                          "- materiel : poste, écran, imprimante, souris, clavier\n"
                                          "- logiciel : application (Outlook, Teams, ERP), mise à jour, erreur d'un programme\n"
                                          "- acces : compte, mot de passe, MFA, droits, hameçonnage\n"
                                          "- autre : tout le reste\n"
                                          "Réponds uniquement par le nom de la catégorie, en minuscules, sans ponctuation."},
            {"role": "user", "content": "Ticket : {{ticket}}"}],
    commit_message="v2 : définition de chaque catégorie (erreurs VPN / Wi-Fi constatées en J2-03)",
)
print(f"version {v2.version} · labels {v2.labels}")
print(f"v{v2.version} → justesse {evaluer(langfuse.get_prompt(NOM, type='chat', label='staging')):.0%}")
langfuse.flush()

**👀 Ce que vous devez observer :** la justesse de chaque version sur les six cas. Mesuré pendant la préparation (`granite4.1:8b`, 21/10/2026) : **v1 50 %** (VPN classé `logiciel`, hameçonnage et MFA classés `autre`), **v2 100 %**. Dans Langfuse, **Prompts → `helpdesk-tri`** : deux versions, leurs labels, et pour chacune les générations liées. Dans **Tracing**, filtrez par **Version** (`helpdesk-tri-v1` / `-v2`) pour comparer les traces.

> ⚠️ **Six cas ne prouvent rien.** Ils suffisent à détecter une régression grossière, pas à affirmer qu'une version est meilleure. La décision de promotion se prend sur le dataset complet (J4-05), idéalement en CI (hors programme de cette session).

## 6. Promouvoir, puis revenir en arrière

Promouvoir, c'est **déplacer le label** `production` : aucun code ne change, le prochain `get_prompt(label="production")` renvoie la v2 (après expiration du cache du SDK).

In [ ]:
langfuse.update_prompt(name=NOM, version=v2.version, new_labels=["production"])
print("production →", langfuse.get_prompt(NOM, type="chat", label="production", cache_ttl_seconds=0).version)

# Incident en production ? Retour arrière : on remet le label sur la v1
langfuse.update_prompt(name=NOM, version=v1.version, new_labels=["production"])
print("retour arrière : production →", langfuse.get_prompt(NOM, type="chat", label="production", cache_ttl_seconds=0).version)

**👀 Ce que vous devez observer :** `production` pointe sur la v2, puis de nouveau sur la v1. Dans l'interface, l'historique des labels garde la trace de ces mouvements.

> 💡 **Le « Tips » du Formateur :** dans cette démonstration, on remet la v1 en production pour montrer le retour arrière. Si votre mesure a montré que la v2 est meilleure, remettez-la en production avant de passer à la suite.

## 7. Et si Langfuse est injoignable ?

Le SDK met les prompts en **cache** (`cache_ttl_seconds`, 60 s par défaut) et accepte un **repli** (`fallback`) : si Langfuse ne répond pas et que rien n'est en cache, le code continue avec le prompt de secours, signalé par `is_fallback`.

In [ ]:
from langfuse import Langfuse

PROMPT_DE_SECOURS = [{"role": "system", "content": "Classe le ticket parmi : {{categories}}. Un seul mot."},
                     {"role": "user", "content": "Ticket : {{ticket}}"}]

injoignable = Langfuse(public_key="pk-lf-injoignable", secret_key="sk-lf-injoignable",
                       base_url="http://127.0.0.1:9", tracing_enabled=False)   # port fermé : simule une panne
p = injoignable.get_prompt(NOM, type="chat", fallback=PROMPT_DE_SECOURS, max_retries=0, fetch_timeout_seconds=2)
print("repli utilisé :", p.is_fallback, "|", p.compile(categories=CATEGORIES, ticket="VPN coupé")[0]["content"])

**👀 Ce que vous devez observer :** `repli utilisé : True`. Le help-desk continue de fonctionner avec un prompt minimal ; une génération faite avec un prompt de repli n'est pas liée à une version, et c'est **visible** dans les traces.

## 8. Gouvernance : qui change un prompt de production ?

| Question | Réponse proposée pour NordTech |
|---|---|
| Qui crée une version ? | les développeurs du help-desk, avec un `commit_message` qui dit **pourquoi** |
| Qui promeut en `production` ? | le responsable du service, après mesure sur le dataset (J4-05) |
| Avec quel test ? | une experiment sans régression sur les scores clés ; une **gate de CI** peut automatiser ce contrôle (hors programme de cette session) |
| Comment revenir en arrière ? | déplacer le label (et le noter dans le registre des changements) |
| Que trace-t-on ? | chaque génération porte sa version : on sait ce qui tournait au moment d'un incident (NIS2 : traçabilité) |

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Créer / versionner un prompt | `langfuse.create_prompt(name=…, type="chat", prompt=[…], labels=[…], commit_message=…)` |
| Le récupérer | `langfuse.get_prompt(nom, type="chat", label="production")`, `.compile(**variables)` |
| L'utiliser dans LangChain, lié aux traces | `ChatPromptTemplate.from_messages(p.get_langchain_prompt())` + `metadata={"langfuse_prompt": p}` |
| Promouvoir / revenir en arrière | `langfuse.update_prompt(name=…, version=…, new_labels=["production"])` |
| Survivre à une panne | `get_prompt(…, fallback=…)`, `cache_ttl_seconds` |

### ✅ Quiz express

1. Que se passe-t-il dans le code quand on promeut une version en `production` ?
2. Pourquoi lier chaque génération à sa version de prompt ?
3. Six cas montrent que la v2 est meilleure : peut-on promouvoir ?
4. Que garantit `fallback` ?

<details>
<summary>Réponses</summary>

1. Rien : seul le label bouge ; le prochain `get_prompt(label="production")` (après le cache) renvoie la nouvelle version.
2. Pour savoir quelle version a produit une réponse (incident, audit), et comparer latence et scores par version.
3. Pas sur six cas : ils détectent une régression grossière ; la décision se prend sur le dataset complet, idéalement en CI.
4. Que le service continue si Langfuse est injoignable et que rien n'est en cache, avec un prompt minimal signalé par `is_fallback`.
</details>